#  Training Transformer on Fire Detection for Indoor Scenario

## Overview
In this notebook, we train a **Transformer** model on a custom dataset designed for detecting **fire scenarios** in **indoor images**. This dataset represents one of the specific scenarios for our **Mixture of Experts (MoE)** model, where each expert specializes in a different scenario (e.g., fire detection in indoor, indoor, satellite, or far-field environments).

### Key Steps:
- **Dataset**: The model is trained using a **indoor fire detection dataset**, which consists of images with fire-related features captured in close and enclosed space.
- **YOLOv8 Training**: The Transformer model is fine-tuned on this dataset, learning to identify fire-related objects.
- **Scenario Expert**: This trained model acts as an expert specifically for detecting fire in indoor images and is part of a broader MoE-based approach for multi-scenario detection.



In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Current device:", torch.cuda.get_device_name(0))

GPU available: True
Current device: NVIDIA GeForce RTX 3070


In [ ]:
# All necessary imports
from torch.utils.data import Dataset, DataLoader
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import torch
import torch.nn as nn
from torchvision.models.detection import FasterRCNN
from torchvision.ops import MultiScaleRoIAlign
from torchvision.models.detection.rpn import AnchorGenerator
from timm import create_model
from torch.optim import AdamW
from tqdm import tqdm

# Dataset (same as outdoor, just folder path changes)
class FireDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
        self.transform = transform

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, os.path.splitext(self.image_files[idx])[0] + '.txt')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        boxes = []
        labels = []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts)
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    x2 = (cx + bw / 2) * w
                    y2 = (cy + bh / 2) * h
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)

        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        if self.transform:
            augmented = self.transform(image=image, bboxes=boxes, labels=labels)
            image = augmented['image']
            boxes = augmented['bboxes']
            labels = augmented['labels']

        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)
        }

        return image, target

    def __len__(self):
        return len(self.image_files)

# Swin Backbone and Identity Transform — unchanged
class SwinBackbone(nn.Module):
    def __init__(self, out_channels=256):
        super().__init__()
        self.body = create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
        self.out_channels = out_channels
        self.fpn_input_channels = [96, 192, 384, 768]
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_c, out_channels, kernel_size=1) for in_c in self.fpn_input_channels
        ])

    def forward(self, x):
        feats = self.body(x)
        fpn_feats = {}
        for idx, feat in enumerate(feats):
            if feat.dim() == 4 and feat.shape[-1] == self.fpn_input_channels[idx]:
                feat = feat.permute(0, 3, 1, 2)
            fpn_feats[str(idx)] = self.lateral_convs[idx](feat)
        return fpn_feats

class IdentityTransform(nn.Module):
    def forward(self, images, targets=None):
        from torchvision.models.detection.image_list import ImageList
        batch_images = torch.stack(images) if isinstance(images, list) else images
        image_sizes = [(224, 224) for _ in range(batch_images.shape[0])]
        image_list = ImageList(batch_images, image_sizes)
        return (image_list, targets) if targets else image_list

    def postprocess(self, result, image_shapes, original_image_sizes):
        return result

# Build model
def get_swin_fasterrcnn(num_classes=2):
    backbone = SwinBackbone(out_channels=256)
    anchor_generator = AnchorGenerator(
        sizes=((16, 32), (32, 64), (64, 128), (128, 256)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 4
    )
    roi_pooler = MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(backbone, num_classes, anchor_generator, roi_pooler, transform=None)
    model.transform = IdentityTransform()
    return model

# Preprocessing
transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# Indoor-specific paths
train_dataset = FireDetectionDataset("indoor/train/images", "indoor/train/labels", transform)
val_dataset = FireDetectionDataset("indoor/valid/images", "indoor/valid/labels", transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))

# Training setup
model = get_swin_fasterrcnn(num_classes=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Training loop
num_epochs = 10
model.train()

print("Training indoor expert...")
for epoch in range(num_epochs):
    total_loss = 0
    successful_batches = 0

    for batch_idx, (images, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")):
        try:
            images = torch.stack(images).to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            successful_batches += 1

        except Exception as e:
            print(f"Error in batch {batch_idx}: {e}")
            continue

    avg_loss = total_loss / max(successful_batches, 1)
    print(f"Epoch {epoch+1}/{num_epochs} - Avg Loss: {avg_loss:.4f} - Successful Batches: {successful_batches}")

# Save model
torch.save(model.state_dict(), "swin_frcnn_indoor.pt")
print("✅ Indoor expert saved as swin_frcnn_indoor.pt")


##  Inference on Indoor Test Images using Trained YOLOv8 Model


In [ ]:
import matplotlib.pyplot as plt
import torchvision
from torchvision.ops import box_iou
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

# Load model and weights
model = get_swin_fasterrcnn(num_classes=2)
model.load_state_dict(torch.load("swin_frcnn_indoor.pt", map_location=device))
model.to(device)
model.eval()

# Define test dataset
test_dataset = FireDetectionDataset("indoor/test/images", "indoor/test/labels", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


iou_threshold = 0.5
all_preds = []
all_gts = []
TP = FP = FN = 0
ious = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Evaluating"):
        images = torch.stack(images).to(device)
        outputs = model(images)
        
        pred_boxes = outputs[0]['boxes'].cpu()
        pred_scores = outputs[0]['scores'].cpu()
        gt_boxes = targets[0]['boxes'].cpu()
        gt_labels = targets[0]['labels'].cpu()
        
        # Filter predictions by score threshold
        score_thresh = 0.5
        keep = pred_scores >= score_thresh
        pred_boxes = pred_boxes[keep]
        pred_scores = pred_scores[keep]

        # Match GT and predicted boxes via IoU
        if len(pred_boxes) > 0 and len(gt_boxes) > 0:
            iou = box_iou(pred_boxes, gt_boxes)
            ious.extend(iou.max(dim=1).values.numpy().tolist())

            for i in range(len(pred_boxes)):
                if iou[i].max().item() >= iou_threshold:
                    TP += 1
                else:
                    FP += 1
            FN += len(gt_boxes) - TP
        elif len(gt_boxes) > 0:
            FN += len(gt_boxes)
        elif len(pred_boxes) > 0:
            FP += len(pred_boxes)
            
            
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
avg_iou = np.mean(ious) if ious else 0.0

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Average IoU: {avg_iou:.4f}")
